# Step 06 — Rule-Based Building Classification

**Input:** `data/output/05_condensed_buildings_with_pois.gpkg`
**Output:** `data/output/06_classified_buildings.gpkg`

Classifies every building with `rule_utils.classify_building()` — a fully deterministic, tag-based classifier. Every answer comes from an explicit lookup table: one row per tag value, and the row *is* the rule. No scoring, no inference, no free-text business names.

**Three layers, strict precedence — the first layer that answers, wins:**

1. **OSM POI tags on the building** — `amenity` / `shop` / `tourism` / `building` / `information`, plus `office=*`, `craft=*`, `social_facility=*` recovered from `additional_information`. Several POIs on one building are each looked up and then reconciled.
2. **ALKIS building-function code** — `function` (e.g. `31001_2000`) against `rule_utils.ALKIS_RULES`, one row per code. Keyed on the code, never the English label: `31001_3031` is *Schloss* but ships as "Lock", and 15 of 301 codes share a label.
3. **OSM footprint type / land use** — for buildings with neither a POI nor an ALKIS code.

An explicit "no activity" answer (e.g. a dwelling) terminates classification — later layers are not consulted.

**Produces, per building:**
- `mid_label` — MiD activity labels (work, school, retail_daily, …)
- `bosserhof_class` — building-use class supplying the worker-density coefficient for notebook 07
- `interpreted_type` — which layer resolved it (`poi_tags` / `alkis_code` / `osm_fallback` / `no_signal`)
- `reason` — machine-readable note on the resolution path

Edit rules in `rule_utils.py`; check what they predict for a given tag set with `python scripts/rule_smoke_test.py`.

> **Interpretation caveat.** ALKIS code `31001_2000` ("Gebäude für Wirtschaft oder Gewerbe") covers 42.6 % of this study area and 82.0 % of *all* non-residential codes state-wide, with a median volume of 126 m³. The specific code for a garage (`31001_2463`) exists but is used 0 times in 4,878,052 buildings, so outbuildings sit in that bucket alongside real commercial premises. No rule can separate them from the code alone. This is left uncorrected on purpose so the ceiling is measured rather than hidden — see `rule_utils.py`.

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd

from rule_utils import classify_building

print('Config loaded')

Config loaded


## 1. Load condensed buildings and classify

In [2]:
buildings = gpd.read_file(CONDENSED_BUILDINGS_FILE)
print(f'Loaded {len(buildings):,} buildings')

results = [classify_building(row) for _, row in buildings.iterrows()]
predictions = pd.DataFrame(results)
print(f'Classified {len(predictions):,} buildings')
predictions.head()

Loaded 574,435 buildings
Classified 574,435 buildings


,gml_id,interpreted_type,mid_labels,bosserhof_class,reason
0,0,alkis_code,"[business, work]",services,rule_source=alkis_code
1,1,alkis_code,"[business, work]",services,rule_source=alkis_code
2,2,alkis_code,"[business, work]",services,rule_source=alkis_code
3,3,alkis_code,"[business, work]",services,rule_source=alkis_code
4,4,alkis_code,"[business, work]",services,rule_source=alkis_code


## 2. Attach geometry and volume, finalize schema

In [3]:
predictions['gml_id'] = predictions['gml_id'].astype(str)
buildings['gml_id'] = buildings['gml_id'].astype(str)

classified = predictions.merge(buildings[['gml_id', 'volume_m3', 'geometry']], on='gml_id', how='left')
classified = gpd.GeoDataFrame(classified, geometry='geometry', crs=buildings.crs)

# Match the column name notebook 07 (redistribution) expects
classified = classified.rename(columns={'mid_labels': 'mid_label'})
classified['mid_label'] = classified['mid_label'].apply(lambda labels: str(list(labels)))

n_cls = classified['bosserhof_class'].notna().sum()
print(f'Classified (has a bosserhof_class): {n_cls:,} / {len(classified):,}  ({100*n_cls/len(classified):.1f}%)')
print()
print('Which layer resolved each building:')
vc = classified['interpreted_type'].value_counts()
for k, v in vc.items():
    print(f'  {k:14s} {v:>8,}  ({100*v/len(classified):5.1f}%)')

Classified (has a bosserhof_class): 282,017 / 574,435  (49.1%)

Which layer resolved each building:
  alkis_code      517,939  ( 90.2%)
  no_signal        43,421  (  7.6%)
  poi_tags         12,485  (  2.2%)
  osm_fallback        590  (  0.1%)


## 3. Save

In [4]:
classified.to_file(CLASSIFIED_BUILDINGS_FILE, driver='GPKG')
print(f'Saved {len(classified):,} rows -> {CLASSIFIED_BUILDINGS_FILE}')

Saved 574,435 rows -> C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\output\06_classified_buildings.gpkg
